# Milestone 2 — Data Cleaning and Preparation
Kenyan retail capstone. Re-runnable from the raw Excel file. Every step records a before/after count in `log`.
Raw data is **synthetic** (see `docs/data_inventory.md`).

In [1]:
import os, sys, subprocess
import pandas as pd, numpy as np
try:
    import holidays
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "holidays"]); import holidays

# Works in VS Code (run from repo root or notebooks/) and in Colab (upload the xlsx next to the notebook)
RAW_NAME = "kenyan_retail_capstone_raw_dataset.xlsx"
CANDIDATES = [f"data/raw/{RAW_NAME}", f"../data/raw/{RAW_NAME}", RAW_NAME]
RAW = next((p for p in CANDIDATES if os.path.exists(p)), None)
assert RAW, "Raw file not found - put it in data/raw/ (or upload it in Colab)"
OUT = "data/clean" if os.path.isdir("data") else ("../data/clean" if os.path.isdir("../data") else ".")
os.makedirs(OUT, exist_ok=True)
log = []   # (step, issue checked, rows affected, action taken)
def note(step, issue, n, action): log.append((step, issue, int(n), action))

## 1. Load the raw sheets

In [2]:
sales = pd.read_excel(RAW, "Raw_Sales")
customers = pd.read_excel(RAW, "Customers")
stores = pd.read_excel(RAW, "Stores")
products = pd.read_excel(RAW, "Products")
for n, d in [("sales", sales), ("customers", customers), ("stores", stores), ("products", products)]:
    print(f"{n:10s} rows={len(d):6d} cols={d.shape[1]}")
raw_rows = len(sales)

sales      rows= 10035 cols=20
customers  rows=  1200 cols=5
stores     rows=    18 cols=6
products   rows=    24 cols=4


## 1b. Audit of the RAW data (before cleaning)
Shows data types, missing values per column, duplicate rows and basic ranges, so the starting point is on record.

In [3]:
def audit(df, name, key=None):
    print(f"===== {name}: {df.shape[0]} rows x {df.shape[1]} columns =====")
    print("Duplicate rows:", df.duplicated().sum(), "| Duplicate keys:", (df.duplicated(key).sum() if key else "n/a"))
    out = pd.DataFrame({"dtype": df.dtypes.astype(str), "missing": df.isna().sum(), "missing_%": (df.isna().mean()*100).round(2), "unique": df.nunique()})
    print(out.to_string(), "\n")
    return out

raw_audit = audit(sales, "Raw_Sales (raw)", ["transaction_id", "line_number"])
audit(customers, "Customers (raw)", "customer_id"); audit(stores, "Stores (raw)", "store_id"); audit(products, "Products (raw)", "product_id")
print(sales[["quantity", "unit_price_kes", "gross_sales_kes", "discount_rate", "net_sales_kes"]].describe().round(2))

===== Raw_Sales (raw): 10035 rows x 20 columns =====
Duplicate rows: 0 | Duplicate keys: 0
                               dtype  missing  missing_%  unique
transaction_id                   str        0       0.00    5000
transaction_datetime  datetime64[us]        0       0.00    4958
transaction_date      datetime64[us]        0       0.00     365
line_number                    int64        0       0.00       4
store_id                         str        0       0.00      18
store_name                       str        0       0.00      18
county                           str        0       0.00       8
town                             str        0       0.00      18
customer_id                      str        0       0.00    1176
product_id                       str        0       0.00      24
product_name                     str        0       0.00      24
product_category                 str        0       0.00       9
quantity                       int64        0       0.00       6

## 2. Types and text

In [4]:
sales["transaction_datetime"] = pd.to_datetime(sales["transaction_datetime"])
sales["transaction_date"] = pd.to_datetime(sales["transaction_date"])
stores["opening_date"] = pd.to_datetime(stores["opening_date"])
# trim whitespace / unify text columns
for name, df in [("sales", sales), ("customers", customers), ("stores", stores), ("products", products)]:
    changed = 0
    for c in df.select_dtypes(include=["object", "string"]).columns:
        before = df[c].copy()
        df[c] = df[c].astype("string").str.strip()
        changed += (before.fillna("") != df[c].fillna("")).sum()
    note("types", f"text cells with stray whitespace ({name})", changed, "stripped")
note("types", "date columns parsed to datetime", len(sales), "converted")
note("types", "date vs datetime mismatch", (sales.transaction_datetime.dt.normalize() != sales.transaction_date).sum(), "none needed")

## 3. Duplicates and keys

In [5]:
note("duplicates", "fully duplicated rows", sales.duplicated().sum(), "none to drop")
note("duplicates", "duplicate (transaction_id, line_number)", sales.duplicated(["transaction_id","line_number"]).sum(), "none to drop")
sales = sales.drop_duplicates(["transaction_id","line_number"])
for name, df, key in [("customers", customers, "customer_id"), ("stores", stores, "store_id"), ("products", products, "product_id")]:
    note("duplicates", f"duplicate {key} in {name}", df.duplicated(key).sum(), "none to drop")
# referential integrity
note("keys", "sales with unknown customer_id", (~sales.customer_id.isin(customers.customer_id)).sum(), "none")
note("keys", "sales with unknown store_id", (~sales.store_id.isin(stores.store_id)).sum(), "none")
note("keys", "sales with unknown product_id", (~sales.product_id.isin(products.product_id)).sum(), "none")
note("keys", "customers with no purchases in 2025", (~customers.customer_id.isin(sales.customer_id)).sum(), "kept in customer table (recency = NaN)")

## 4. Missing values

In [6]:
miss = sales.isna().sum()
print(miss[miss > 0])
note("missing", "promotion is blank (means no promotion, not an error)", sales.promotion.isna().sum(), "filled with 'No promotion'")
sales["promotion"] = sales["promotion"].fillna("No promotion")
assert sales.drop(columns=[]).isna().sum().sum() == 0, "unexpected missing values remain"

promotion    5657
dtype: int64


## 5. Business-rule checks (do the money columns agree?)

In [7]:
tol = 0.02
bad_gross = (sales.quantity * sales.unit_price_kes - sales.gross_sales_kes).abs() > tol
bad_disc  = (sales.gross_sales_kes * sales.discount_rate - sales.discount_amount_kes).abs() > tol
bad_net   = (sales.gross_sales_kes - sales.discount_amount_kes - sales.net_sales_kes).abs() > tol
note("rules", "gross != quantity x unit_price", bad_gross.sum(), "all consistent")
note("rules", "discount_amount != gross x discount_rate", bad_disc.sum(), "all consistent")
note("rules", "net != gross - discount", bad_net.sum(), "all consistent")
note("rules", "quantity <= 0 or price <= 0", ((sales.quantity <= 0) | (sales.unit_price_kes <= 0)).sum(), "none")
# store / product names agree with the master tables?
m = sales.merge(stores[["store_id","store_name","county"]], on="store_id", suffixes=("","_m"))
note("rules", "store name/county differs from Stores table", ((m.store_name != m.store_name_m) | (m.county != m.county_m)).sum(), "none")
m = sales.merge(products, on="product_id", suffixes=("","_m"))
note("rules", "product name/category differs from Products table", ((m.product_name != m.product_name_m) | (m.product_category != m.category)).sum(), "none")
# promotion label vs real discount
sales["promo_label"] = sales.promotion != "No promotion"
sales["discount_applied"] = sales.discount_rate > 0
note("rules", "promotion label present but discount = 0%", (sales.promo_label & ~sales.discount_applied).sum(),
     "kept; new column discount_applied (use this, not the label, to judge promotion effect)")

## 6. Outliers (flag, do not delete)

In [8]:
# Compare each line with its own product category (a TV and a loaf of bread should not share one threshold)
q = sales.groupby("product_category")["net_sales_kes"]
q1, q3 = q.transform(lambda s: s.quantile(.25)), q.transform(lambda s: s.quantile(.75))
sales["is_outlier_line"] = sales.net_sales_kes > (q3 + 1.5 * (q3 - q1))
# unit price vs the product's standard price
sales = sales.merge(products[["product_id","standard_unit_price_kes"]], on="product_id", how="left")
sales["price_ratio"] = sales.unit_price_kes / sales.standard_unit_price_kes
sales["is_price_outlier"] = ~sales.price_ratio.between(0.8, 1.2)
note("outliers", "line value above category IQR fence", sales.is_outlier_line.sum(),
     "flagged only: large baskets are real behaviour, not errors")
note("outliers", "unit price >20% from standard price", sales.is_price_outlier.sum(), "none; prices stay within about -8% to +12%")
print(sales[["net_sales_kes","price_ratio"]].describe().round(2))

       net_sales_kes  price_ratio
count       10035.00     10035.00
mean          670.28         1.02
std           882.79         0.06
min            57.31         0.92
25%           216.29         0.97
50%           381.86         1.02
75%           805.14         1.07
max          9685.98         1.12


## 7. Public context: Kenyan public holidays and CPI

In [9]:
ke = holidays.KE(years=[2025])         # python-holidays package, MIT licence
hol = pd.DataFrame({"transaction_date": pd.to_datetime(list(ke.keys())), "holiday_name": list(ke.values())})
sales = sales.merge(hol, on="transaction_date", how="left")
sales["is_public_holiday"] = sales.holiday_name.notna()
note("context", "lines on a Kenyan public holiday", sales.is_public_holiday.sum(), "joined holiday flag + name")

# CPI: fill data/external/kenya_cpi_2025.csv from KNBS (see docs/data_inventory.md), then re-run
cpi_path = next((p for p in ["data/external/kenya_cpi_2025.csv","../data/external/kenya_cpi_2025.csv"] if os.path.exists(p)), None)
sales["month"] = sales.transaction_date.dt.to_period("M").astype(str)
if cpi_path:
    cpi = pd.read_csv(cpi_path)
    if cpi["cpi_index"].notna().all():
        sales = sales.merge(cpi[["month","cpi_index","inflation_yoy_pct"]], on="month", how="left")
        note("context", "lines joined to monthly KNBS CPI", sales.cpi_index.notna().sum(), "joined")
    else:
        note("context", "CPI file present but still blank", 0, "TODO: fill from KNBS, then re-run")
else:
    note("context", "CPI file missing", 0, "TODO")

## 8. Feature engineering

In [10]:
d = sales.transaction_datetime
sales["year_month"] = sales.month
sales["week_start"] = (sales.transaction_date - pd.to_timedelta(sales.transaction_date.dt.weekday, unit="D"))
sales["weekday"] = d.dt.day_name()
sales["is_weekend"] = d.dt.weekday >= 5
sales["hour"] = d.dt.hour
sales["is_mpesa"] = sales.payment_method == "M-Pesa"
sales["is_month_end_week"] = d.dt.day >= 25     # pay-day effect worth testing
sales = sales.merge(customers, on="customer_id", how="left").merge(stores[["store_id","store_type","opening_date"]], on="store_id", how="left")
sales["store_age_years"] = ((sales.transaction_date - sales.opening_date).dt.days / 365.25).round(1)
sales["customer_in_store_county"] = sales.customer_county == sales.county
note("features", "new columns added", 0, "calendar, weekend, hour, M-Pesa flag, store age/type, customer segment, loyalty, local-customer flag")

# Transaction (basket) level table: one row per transaction
basket = (sales.groupby("transaction_id").agg(
    transaction_datetime=("transaction_datetime","first"), store_id=("store_id","first"), county=("county","first"),
    customer_id=("customer_id","first"), payment_method=("payment_method","first"),
    n_lines=("line_number","count"), units=("quantity","sum"),
    gross_sales_kes=("gross_sales_kes","sum"), discount_kes=("discount_amount_kes","sum"),
    net_sales_kes=("net_sales_kes","sum"), any_discount=("discount_applied","max")).reset_index())
basket["year_month"] = basket.transaction_datetime.dt.to_period("M").astype(str)
note("features", "transaction-level rows (baskets)", len(basket), "built basket table")

# Customer level table (RFM inputs). Snapshot date = day after the last sale in the data.
snap = sales.transaction_date.max() + pd.Timedelta(days=1)
cust = (basket.groupby("customer_id").agg(last_purchase=("transaction_datetime","max"),
        frequency=("transaction_id","nunique"), monetary_kes=("net_sales_kes","sum"),
        avg_basket_kes=("net_sales_kes","mean")).reset_index())
cust["recency_days"] = (snap - cust.last_purchase.dt.normalize()).dt.days
cust = customers.merge(cust, on="customer_id", how="left")
note("features", "customer-level rows (incl. 24 with no purchases)", len(cust), "built customer table")

## 8b. Audit of the CLEAN data (after cleaning)
Same checks as 1b, plus a side-by-side before/after comparison.

In [11]:
clean_audit = audit(sales, "Sales (clean)", ["transaction_id", "line_number"])
raw_cols = list(raw_audit.index)   # the 20 original columns
compare = pd.DataFrame({
    "metric": ["rows", "columns", "missing cells", "duplicate rows", "duplicate (txn, line)", "transactions", "customers seen"],
    "raw": [raw_rows, 20, int(raw_audit["missing"].sum()), 0, 0, 5000, 1176],
    "clean": [len(sales), sales.shape[1], int(sales[raw_cols].isna().sum().sum()),
              int(sales.duplicated().sum()), int(sales.duplicated(["transaction_id","line_number"]).sum()),
              sales.transaction_id.nunique(), sales.customer_id.nunique()]})
print(compare.to_string(index=False))
# Columns with blanks in the clean data are blank ON PURPOSE:
print("\nBlanks left in clean data (expected):")
print(sales.isna().sum()[lambda x: x > 0].to_string(), "\n-> holiday_name is empty on non-holidays; cpi columns stay empty until the KNBS file is filled")

===== Sales (clean): 10035 rows x 44 columns =====


Duplicate rows: 0 | Duplicate keys: 0
                                   dtype  missing  missing_%  unique
transaction_id                    string        0       0.00    5000
transaction_datetime      datetime64[us]        0       0.00    4958
transaction_date          datetime64[us]        0       0.00     365
line_number                        int64        0       0.00       4
store_id                          string        0       0.00      18
store_name                        string        0       0.00      18
county                            string        0       0.00       8
town                              string        0       0.00      18
customer_id                       string        0       0.00    1176
product_id                        string        0       0.00      24
product_name                      string        0       0.00      24
product_category                  string        0       0.00       9
quantity                           int64        0       0.00     

## 9. Final checks, before/after counts, save

In [12]:
assert sales.transaction_id.nunique() == 5000 and len(sales) == raw_rows, "row counts changed unexpectedly"
assert (sales.net_sales_kes > 0).all()
log_df = pd.DataFrame(log, columns=["step","check","rows_affected","action"])
display_cols = log_df[log_df.rows_affected > 0]
print(f"Rows in: {raw_rows}   Rows out: {len(sales)}   Columns out: {sales.shape[1]}")
print("Total net sales (KES):", round(sales.net_sales_kes.sum(), 2))
sales.drop(columns=["opening_date"]).to_csv(f"{OUT}/sales_clean.csv", index=False)
basket.to_csv(f"{OUT}/baskets_clean.csv", index=False)
cust.to_csv(f"{OUT}/customers_clean.csv", index=False)
stores.to_csv(f"{OUT}/stores_clean.csv", index=False)
products.to_csv(f"{OUT}/products_clean.csv", index=False)
log_df.to_csv(f"{OUT}/cleaning_log.csv", index=False)
log_df

Rows in: 10035   Rows out: 10035   Columns out: 44
Total net sales (KES): 6726287.91


,step,check,rows_affected,action
0,types,text cells with stray whitespace (sales),0,stripped
1,types,text cells with stray whitespace (customers),0,stripped
2,types,text cells with stray whitespace (stores),0,stripped
3,types,text cells with stray whitespace (products),0,stripped
4,types,date columns parsed to datetime,10035,converted
5,types,date vs datetime mismatch,0,none needed
6,duplicates,fully duplicated rows,0,none to drop
7,duplicates,"duplicate (transaction_id, line_number)",0,none to drop
8,duplicates,duplicate customer_id in customers,0,none to drop
9,duplicates,duplicate store_id in stores,0,none to drop
